In [12]:
import pandas as pd
import numpy
from datetime import datetime

current_year = datetime.now().year
previous_year = current_year - 1

In [24]:
jar = pd.read_csv("./data/JAR_IREGISTRUOTI.csv", delimiter="|", usecols=["ja_kodas", "ja_pavadinimas", "ja_reg_data", "form_pavadinimas", "stat_pavadinimas", "stat_data_nuo"] )
jar["stat_pavadinimas"] = jar["stat_pavadinimas"].replace("Teisinis stat neįregistruotas", "Aktyvus")

nvo = pd.read_csv("./data/JAR_NVO_NUO.csv", delimiter="|", usecols=["ja_kodas", "nvo_nuo"])
nvo["nvo_nuo"] = pd.to_datetime(nvo["nvo_nuo"])
nvo_idx = nvo.groupby("ja_kodas")["nvo_nuo"].idxmax()
nvo = nvo.loc[nvo_idx].reset_index(drop=True)

nvo["nvo_statusas"] = 1

par_gav = pd.read_csv("./data/JAR_PARAMOS_GAV_NUO.csv", delimiter="|", usecols=["ja_kodas", "paramos_gav_nuo"])
par_gav["paramos_gav_nuo"] = pd.to_datetime(par_gav["paramos_gav_nuo"])
par_gav_idx = par_gav.groupby("ja_kodas")["paramos_gav_nuo"].idxmax()
par_gav = par_gav.loc[par_gav_idx].reset_index(drop=True)
par_gav["par_gav_statusas"] = 1


In [25]:
ex_jar = pd.read_csv("./data/JAR_ISREGISTRUOTI.csv", delimiter="|", usecols=["ja_kodas", "ja_pavadinimas", "ja_reg_data", "form_pavadinimas", "isreg_data"])
ex_jar["stat_pavadinimas"] = "Išregistruotas"
ex_jar = ex_jar.rename(columns={"isreg_data": "stat_data_nuo"})


# ex_nvo = pd.read_csv("./data/JAR_NVO_IKI.csv", delimiter="|", usecols=["ja_kodas", "nvo_nuo", "nvo_iki"])
# ex_nvo["nvo_statusas"] = 0

# ex_par_gav = pd.read_csv("./data/JAR_PARAMOS_GAV_IKI.csv", delimiter="|", usecols=["ja_kodas", "paramos_gav_nuo", "paramos_gav_iki"])
# ex_par_gav["par_gav_statusas"] = 0

In [26]:
jar_concat = pd.concat([jar, ex_jar], ignore_index=True)

In [27]:
nepateike_fa = pd.read_csv("./data/JAR_NEPATEIKE_FA_UZ_PRAEJUSIUS.csv", delimiter="|", usecols=["ja_kodas", "fa_nepateikta_uz_metus"])
nepateike_fa = nepateike_fa[nepateike_fa["fa_nepateikta_uz_metus"] == previous_year]
nepateike_fa["atskaitingas"] = 0

In [31]:
df_merged = jar_concat.merge(nvo, on="ja_kodas", how="left")
df_merged = df_merged.merge(par_gav, on="ja_kodas", how="left")
df_merged = df_merged.merge(nepateike_fa, on="ja_kodas", how="left")
df_merged["atskaitingas"] = df_merged["atskaitingas"].fillna(1).astype(int)
df_merged.to_csv("result.csv", index=False)

In [32]:
df_merged["nvo_statusas"] = df_merged["nvo_statusas"].fillna(0).astype(int)
df_merged["par_gav_statusas"] = df_merged["par_gav_statusas"].fillna(0).astype(int)
df_merged["fa_nepateikta_uz_metus"] = df_merged["fa_nepateikta_uz_metus"].fillna(0).astype(int)
df_merged.to_csv("results.csv", index=False)

In [37]:
nvo = pd.read_csv("./data/JAR_NVO_NUO.csv", delimiter="|", usecols=["ja_kodas", "nvo_nuo"])
nvo[nvo.duplicated(subset=["ja_kodas"])]



,ja_kodas,nvo_nuo
2623,255965080,2021-11-04


In [36]:
par_gav = pd.read_csv("./data/JAR_PARAMOS_GAV_NUO.csv", delimiter="|", usecols=["ja_kodas", "paramos_gav_nuo"])
par_gav[par_gav.duplicated(subset=["ja_kodas"])]

,ja_kodas,paramos_gav_nuo
12343,293343340,2004-02-20
